## Example 1: a dataset of cases, an EqualsExpected evaluator, and the evaluation report

In [ ]:
# Example 1: a dataset of cases, an EqualsExpected evaluator, and the evaluation report
import importlib.metadata

from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import EqualsExpected

print(f"pydantic_evals {importlib.metadata.version('pydantic_evals')}")


def double(x: int) -> int:
    return x * 2


dataset = Dataset(
    name="doubling",
    cases=[
        Case(name="two", inputs=2, expected_output=4),
        Case(name="five", inputs=5, expected_output=10),
        # Deliberately wrong expectation: this case must fail its assertion.
        Case(name="negative", inputs=-3, expected_output=6),
    ],
    evaluators=[EqualsExpected()],
)

# Inside Jupyter the event loop is already running, so await evaluate() instead of evaluate_sync().
report = await dataset.evaluate(double, progress=False)

results = {case.name: (case.output, case.assertions["EqualsExpected"].value) for case in report.cases}
print(results)
assert results == {"two": (4, True), "five": (10, True), "negative": (-6, False)}, results
assert report.failures == [], report.failures
# The averages row reports the share of assertions that passed: 2 of 3.
assert abs(report.averages().assertions - 2 / 3) < 1e-9, report.averages()

## Example 2: custom evaluators that return assertions, scores, labels and reasons

In [ ]:
# Example 2: custom evaluators that return assertions, scores, labels and reasons
from dataclasses import dataclass

from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import EvaluationReason, Evaluator, EvaluatorContext


@dataclass
class WordOverlap(Evaluator[str, str]):
    """Score the share of expected words found in the output, and label its length."""

    def evaluate(self, ctx: EvaluatorContext[str, str]):
        got = set(ctx.output.lower().split())
        want = set(ctx.expected_output.lower().split())
        # A bool becomes an assertion, a number a score, and a string a label.
        return {
            "overlap": len(got & want) / len(want),
            "exact": ctx.output == ctx.expected_output,
            "length": "short" if len(ctx.output) < 12 else "long",
        }


@dataclass
class IsUpperCase(Evaluator[str, str]):
    def evaluate(self, ctx: EvaluatorContext[str, str]) -> EvaluationReason:
        return EvaluationReason(value=ctx.output.isupper(), reason=f"output={ctx.output!r}")


def shout(text: str) -> str:
    return text.upper()


dataset = Dataset(
    name="shout",
    cases=[
        Case(name="hello", inputs="hello world", expected_output="HELLO WORLD"),
        Case(name="morning", inputs="good morning to you", expected_output="GOOD MORNING"),
    ],
    evaluators=[WordOverlap(), IsUpperCase()],
)
report = await dataset.evaluate(shout, progress=False)

by_name = {case.name: case for case in report.cases}
for name, case in by_name.items():
    print(name, {k: v.value for k, v in case.assertions.items()}, {k: v.value for k, v in case.scores.items()},
          {k: v.value for k, v in case.labels.items()})

hello, morning = by_name["hello"], by_name["morning"]
assert hello.assertions["exact"].value is True and morning.assertions["exact"].value is False
assert hello.scores["overlap"].value == 1.0 and morning.scores["overlap"].value == 1.0
assert (hello.labels["length"].value, morning.labels["length"].value) == ("short", "long")
assert hello.assertions["IsUpperCase"].value is True
assert hello.assertions["IsUpperCase"].reason == "output='HELLO WORLD'", hello.assertions["IsUpperCase"]

## Example 3: built-in evaluators, at dataset level and on a single case

In [ ]:
# Example 3: built-in evaluators, at dataset level and on a single case
from pydantic_evals import Case, Dataset
from pydantic_evals.evaluators import Contains, Equals, IsInstance, MaxDuration


def describe(n: int) -> str:
    return f"{n} is {'even' if n % 2 == 0 else 'odd'}"


dataset = Dataset(
    name="parity",
    cases=[
        Case(name="four", inputs=4, evaluators=[Contains(value="even"), Equals(value="4 is even")]),
        Case(name="seven", inputs=7, evaluators=[Contains(value="even")]),
    ],
    # Evaluators listed on the dataset run for every case.
    evaluators=[IsInstance(type_name="str"), MaxDuration(seconds=5)],
)
report = await dataset.evaluate(describe, progress=False)

assertions = {case.name: {k: v.value for k, v in case.assertions.items()} for case in report.cases}
print(assertions)
assert assertions["four"] == {"Contains": True, "Equals": True, "IsInstance": True, "MaxDuration": True}, assertions
assert assertions["seven"] == {"Contains": False, "IsInstance": True, "MaxDuration": True}, assertions

## Example 4: task failures, custom metrics and attributes

In [ ]:
# Example 4: task failures, custom metrics and attributes
from pydantic_evals import Case, Dataset, increment_eval_metric, set_eval_attribute
from pydantic_evals.evaluators import EqualsExpected


def parse_price(text: str) -> float:
    # Metrics and attributes recorded while the task runs are attached to the case's report.
    increment_eval_metric("characters", len(text))
    set_eval_attribute("currency", text[:1])
    return float(text.lstrip("$"))


dataset = Dataset(
    name="prices",
    cases=[
        Case(name="whole", inputs="$12", expected_output=12.0),
        Case(name="cents", inputs="$3.50", expected_output=3.5),
        Case(name="garbage", inputs="$abc", expected_output=0.0),
    ],
    evaluators=[EqualsExpected()],
)
report = await dataset.evaluate(parse_price, progress=False)

# A task that raises does not stop the run: the case is reported as a failure instead.
assert [case.name for case in report.cases] == ["whole", "cents"], [case.name for case in report.cases]
assert [failure.name for failure in report.failures] == ["garbage"], report.failures
print("failure:", report.failures[0].error_message)
assert "ValueError" in report.failures[0].error_message, report.failures[0].error_message

whole, cents = report.cases
assert (whole.metrics, cents.metrics) == ({"characters": 3}, {"characters": 5}), (whole.metrics, cents.metrics)
assert whole.attributes == {"currency": "$"}, whole.attributes
assert all(case.assertions["EqualsExpected"].value for case in report.cases)

## Example 5: load a dataset from YAML, save it to JSON with a schema, and run an async task

In [ ]:
# Example 5: load a dataset from YAML, save it to JSON with a schema, and run an async task
import asyncio
import json
import os
import tempfile
from dataclasses import dataclass
from typing import Any

from pydantic_evals import Dataset
from pydantic_evals.evaluators import Evaluator, EvaluatorContext


@dataclass
class StartsWithCapital(Evaluator[str, str, Any]):
    def evaluate(self, ctx: EvaluatorContext[str, str, Any]) -> bool:
        return ctx.output[:1].isupper()


yaml_text = """
name: capitals
cases:
- name: france
  inputs: France
  expected_output: Paris
- name: japan
  inputs: Japan
  expected_output: Tokyo
  metadata: {region: asia}
evaluators:
- EqualsExpected
- StartsWithCapital
"""
# Custom evaluators are referenced by class name and must be passed in when loading.
dataset = Dataset[str, str, Any].from_text(yaml_text, fmt="yaml", custom_evaluator_types=[StartsWithCapital])
assert dataset.name == "capitals" and [case.name for case in dataset.cases] == ["france", "japan"]
assert dataset.cases[1].metadata == {"region": "asia"}, dataset.cases[1].metadata

with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "capitals.json")
    dataset.to_file(path, custom_evaluator_types=[StartsWithCapital])
    # A JSON schema for editor completion is written next to the dataset.
    assert sorted(os.listdir(tmp)) == ["capitals.json", "capitals_schema.json"], os.listdir(tmp)
    with open(path) as f:
        saved = json.load(f)
    reloaded = Dataset[str, str, Any].from_file(path, custom_evaluator_types=[StartsWithCapital])
assert [case["expected_output"] for case in saved["cases"]] == ["Paris", "Tokyo"], saved["cases"]
assert [case.inputs for case in reloaded.cases] == ["France", "Japan"]

CAPITALS = {"France": "Paris", "Japan": "Kyoto"}  # Japan is deliberately wrong


async def lookup(country: str) -> str:
    await asyncio.sleep(0.01)
    return CAPITALS[country]


report = await reloaded.evaluate(lookup, max_concurrency=2, progress=False)
results = {case.name: {k: v.value for k, v in case.assertions.items()} for case in report.cases}
print(results)
assert results == {
    "france": {"EqualsExpected": True, "StartsWithCapital": True},
    "japan": {"EqualsExpected": False, "StartsWithCapital": True},
}, results